# Student Model Evaluation Notebook
This notebook runs evaluation on the student sequence tagger model completely independently of the training notebook.


In [2]:
import json
import pickle
import random
import gc
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import XLMRobertaModel, XLMRobertaTokenizerFast
from torchcrf import CRF
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support, accuracy_score
from sklearn.metrics import homogeneity_score, completeness_score, v_measure_score, matthews_corrcoef, cohen_kappa_score

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', DEVICE)

OUTPUTS_DIR = Path('outputs')
CHECKPOINTS_DIR = OUTPUTS_DIR / 'checkpoints'
FIGURES_DIR = OUTPUTS_DIR / 'figures'
FIGURES_DIR.mkdir(exist_ok=True, parents=True)


Using device: cuda


In [3]:
def print_gpu_memory(stage):
    """Prints current GPU memory usage."""
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / (1024 ** 2)
        reserved = torch.cuda.memory_reserved() / (1024 ** 2)
        max_allocated = torch.cuda.max_memory_allocated() / (1024 ** 2)
        max_reserved = torch.cuda.max_memory_reserved() / (1024 ** 2)
        print("==============================")
        print(f"Stage: {stage}")
        print(f"Allocated : {allocated:.2f} MB")
        print(f"Reserved  : {reserved:.2f} MB")
        print(f"Peak Alloc: {max_allocated:.2f} MB")
        print(f"Peak Res  : {max_reserved:.2f} MB")
        print("==============================")
        print()

def free_gpu_memory(*objects):
    """Deletes objects from globals, collects garbage, empty cache, and checks memory/leaks."""
    freed_names = []
    for obj in objects:
        if isinstance(obj, str):
            if obj in globals():
                freed_names.append(obj)
                del globals()[obj]
        else:
            names = [k for k, v in globals().items() if v is obj]
            if names:
                for name in names:
                    freed_names.append(name)
                    del globals()[name]
            else:
                del obj
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
    print("Freed:")
    if freed_names:
        for name in freed_names:
            print(name)
    else:
        print("None")
    print()
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / (1024 ** 2)
        reserved = torch.cuda.memory_reserved() / (1024 ** 2)
        print(f"Memory after cleanup - Allocated: {allocated:.2f} MB, Reserved: {reserved:.2f} MB")
        print()


In [4]:
class CharCNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim=30, out_channels=50, kernel_sizes=(3, 4, 5)):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.convs = nn.ModuleList([
            nn.Conv1d(embedding_dim, out_channels, k, padding=k // 2)
            for k in kernel_sizes
        ])
        self.dropout = nn.Dropout(0.2)
        self.output_dim = out_channels * len(kernel_sizes)

    def forward(self, x):
        B, S, C = x.shape
        x = x.reshape(B * S, C)
        x = self.embedding(x)  # [B*S, C, E]
        x = x.transpose(1, 2)  # [B*S, E, C]
        conv_outputs = []
        for conv in self.convs:
            y = torch.relu(conv(x))
            y = torch.max(y, dim=2).values
            conv_outputs.append(y)
        x = torch.cat(conv_outputs, dim=1)
        x = self.dropout(x)
        return x.reshape(B, S, -1)

class POSModel(nn.Module):
    def __init__(self, tagset_size, char_vocab_dict, char_embedding_dim=30, char_out=50, lstm_hidden=256):
        super().__init__()
        self.encoder = XLMRobertaModel.from_pretrained('xlm-roberta-base')
        self.char_cnn = CharCNN(len(char_vocab_dict), char_embedding_dim, char_out)
        char_feature_dim = self.char_cnn.output_dim
        self.lstm = nn.LSTM(
            self.encoder.config.hidden_size + char_feature_dim,
            lstm_hidden // 2,
            batch_first=True,
            bidirectional=True
        )
        self.hidden2tag = nn.Linear(lstm_hidden, tagset_size)
        self.crf = CRF(tagset_size, batch_first=True)
        self.char_vocab = char_vocab_dict

    def forward(self, input_ids, attention_mask, labels=None, char_ids=None):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = outputs.last_hidden_state
        char_features = self.char_cnn(char_ids)
        combined = torch.cat([sequence_output, char_features], dim=-1)
        lstm_out, _ = self.lstm(combined)
        emissions = self.hidden2tag(lstm_out)
        if labels is not None:
            labels = labels.clone()
            labels[labels == -100] = 0
            loss = -self.crf(
                emissions,
                labels,
                mask=attention_mask.bool(),
                reduction='mean'
            )
            return loss
        tags = self.crf.decode(emissions, mask=attention_mask.bool())
        return tags


In [5]:
class POSDataset(Dataset):
    def __init__(self, sentences, tokenizer=None, max_length=128):
        self.sentences = sentences
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx):
        sent = self.sentences[idx]
        words = [w for w, _ in sent]
        tags = [t for _, t in sent]
        return {
            "tokens": words,
            "tags": tags
        }

def collate_fn(batch):
    """Collates list of sample dictionaries into torch batches with dynamic padding."""
    batch_words = [item['tokens'] for item in batch]
    batch_tags = [item['tags'] for item in batch]
    
    encoding = tokenizer(
        batch_words,
        is_split_into_words=True,
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt"
    )
    
    batch_labels = []
    for i in range(len(batch)):
        word_ids = encoding.word_ids(batch_index=i)
        labels = []
        prev_wid = None
        tags = batch_tags[i]
        for wid in word_ids:
            if wid is None:
                labels.append(-100)
            elif wid != prev_wid:
                tag = tags[wid] if wid < len(tags) else 'N'
                labels.append(TAG_TO_ID.get(tag, TAG_TO_ID['N']))
            else:
                labels.append(-100)
            prev_wid = wid
        batch_labels.append(torch.tensor(labels, dtype=torch.long))
        
    labels_tensor = torch.stack(batch_labels)
    
    batch_size = len(batch)
    seq_len = encoding['input_ids'].size(1)
    max_word_len = 20
    char_ids = torch.zeros(batch_size, seq_len, max_word_len, dtype=torch.long)
    
    for i in range(batch_size):
        word_ids = encoding.word_ids(batch_index=i)
        words = batch_words[i]
        for j, wid in enumerate(word_ids):
            if wid is not None and wid < len(words):
                word = words[wid]
                for k, ch in enumerate(word[:max_word_len]):
                    char_ids[i, j, k] = char_vocab.get(ch, char_vocab['<unk>'])
                    
    collated = {
        'input_ids': encoding['input_ids'],
        'attention_mask': encoding['attention_mask'],
        'labels': labels_tensor,
        'char_ids': char_ids,
        'tokens': batch_words
    }
    
    del batch_labels, labels_tensor, char_ids
    
    return collated


In [6]:
def evaluate_model(model, loader, device):
    """Evaluates structural tagger outputs on reference dataloaders."""
    model.eval()
    y_true, y_pred = [], []
    total_log_lik = 0.0
    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            char_ids = batch['char_ids'].to(device)
            labels = batch['labels'].to(device)
            
            predictions = model(input_ids, attention_mask, char_ids=char_ids)
            labels_cpu = labels.cpu().tolist()
            
            loss = model(input_ids, attention_mask, labels=labels, char_ids=char_ids)
            total_log_lik += -loss.item()
            
            for pred, labels_row in zip(predictions, labels_cpu):
                gold = [l for l in labels_row if l != -100]
                m = min(len(pred), len(gold))
                y_true.extend(ID_TO_TAG[g] for g in gold[:m])
                y_pred.extend(ID_TO_TAG[p] for p in pred[:m])
            # Free batch tensors immediately
            del input_ids, attention_mask, char_ids, labels, predictions, loss
                
    accuracy = accuracy_score(y_true, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)
    avg_log_lik = total_log_lik / len(loader)
    
    return {
        'accuracy': accuracy,
        'precision': prec,
        'recall': rec,
        'f1': f1,
        'avg_log_likelihood': avg_log_lik,
        'y_true': y_true,
        'y_pred': y_pred
    }


In [7]:
# Load configuration
with open(OUTPUTS_DIR / "pseudo_label_stats.json", 'r') as f:
    config = json.load(f)
print("Training Configuration:")
print(json.dumps(config, indent=2))
print()

MAX_LENGTH = config.get("max_length", 96)
BATCH_SIZE = config.get("batch_size", 2)

# Load tag mappings
with open(OUTPUTS_DIR / "pseudo_labels.pkl", 'rb') as f:
    tag_maps = pickle.load(f)
TAG_TO_ID = tag_maps["TAG_TO_ID"]
ID_TO_TAG = tag_maps["ID_TO_TAG"]
CANONICAL_TAGS = tag_maps["CANONICAL_TAGS"]
print("Loaded tag mappings:", CANONICAL_TAGS)

# Load character vocabulary
with open(OUTPUTS_DIR / "char_vocab.pkl", 'rb') as f:
    char_vocab = pickle.load(f)
print(f"Loaded character vocabulary with {len(char_vocab)} characters.")

# Load tokenizer
tokenizer = XLMRobertaTokenizerFast.from_pretrained(OUTPUTS_DIR / "tokenizer")
print("Loaded Hugging Face Tokenizer.")

# Load validation dataset
with open(OUTPUTS_DIR / "valid_sentences.pkl", 'rb') as f:
    valid_sentences = pickle.load(f)
print(f"Loaded {len(valid_sentences)} validation sentences.")


Training Configuration:
{
  "total": 717,
  "accepted": 80,
  "active_learning": 210,
  "rejected": 427
}



TypeError: list indices must be integers or slices, not str

In [ ]:
best_path = CHECKPOINTS_DIR / "student_best_model.pt"
latest_path = CHECKPOINTS_DIR / "student_latest_model.pt"

if best_path.exists():
    checkpoint_path = best_path
    print(f"Automatically detected and loading best model: {checkpoint_path}")
elif latest_path.exists():
    checkpoint_path = latest_path
    print(f"Automatically detected and loading latest model: {checkpoint_path}")
else:
    raise FileNotFoundError("Error: Neither student_best_model.pt nor student_latest_model.pt exists in checkpoints!")

student_model = POSModel(len(CANONICAL_TAGS), char_vocab).to(DEVICE)
checkpoint = torch.load(checkpoint_path, map_location=DEVICE)

if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
    student_model.load_state_dict(checkpoint["model_state_dict"])
else:
    student_model.load_state_dict(checkpoint)

student_model.eval()
print("Student model successfully loaded and set to evaluation mode.")


In [ ]:
student_valid_dataset = POSDataset(valid_sentences, tokenizer)
student_valid_loader = DataLoader(student_valid_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

print_gpu_memory("Before Evaluation")
final_metrics = evaluate_model(student_model, student_valid_loader, DEVICE)
print_gpu_memory("After Evaluation")

y_true = final_metrics["y_true"]
y_pred = final_metrics["y_pred"]

accuracy = final_metrics['accuracy']
precision = final_metrics['precision']
recall = final_metrics['recall']
macro_f1 = final_metrics['f1']
avg_log_lik = final_metrics['avg_log_likelihood']

homogeneity = homogeneity_score(y_true, y_pred)
completeness = completeness_score(y_true, y_pred)
v_measure = v_measure_score(y_true, y_pred)
mcc = matthews_corrcoef(y_true, y_pred)
kappa = cohen_kappa_score(y_true, y_pred)

print("=" * 60)
print("Student Model Final Metrics:")
print(f"Accuracy              : {accuracy:.4f}")
print(f"Precision (Macro)     : {precision:.4f}")
print(f"Recall (Macro)        : {recall:.4f}")
print(f"Macro F1              : {macro_f1:.4f}")
print(f"Avg Log-Likelihood    : {avg_log_lik:.4f}")
print(f"Homogeneity           : {homogeneity:.4f}")
print(f"Completeness          : {completeness:.4f}")
print(f"V-Measure             : {v_measure:.4f}")
print(f"Matthews Corr. Coef.  : {mcc:.4f}")
print(f"Cohen's Kappa         : {kappa:.4f}")
print("=" * 60)

# Save outputs/final_metrics.json
final_metrics_dict = {
    'accuracy': accuracy,
    'precision': precision,
    'recall': recall,
    'f1': macro_f1,
    'avg_log_likelihood': avg_log_lik,
    'homogeneity': homogeneity,
    'completeness': completeness,
    'v_measure': v_measure,
    'matthews_corrcoef': mcc,
    'cohen_kappa': kappa
}
with open(OUTPUTS_DIR / 'final_metrics.json', 'w') as f:
    json.dump(final_metrics_dict, f, indent=2)
print("Saved outputs/final_metrics.json")


In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=CANONICAL_TAGS)
df_cm = pd.DataFrame(cm, index=CANONICAL_TAGS, columns=CANONICAL_TAGS)
df_cm.to_csv(OUTPUTS_DIR / "confusion_matrix.csv")
print("Saved outputs/confusion_matrix.csv")

plt.figure(figsize=(10, 8))
sns.heatmap(df_cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Validation POS Confusion Matrix')
plt.savefig(FIGURES_DIR / 'confusion_matrix.png', bbox_inches='tight')
plt.show()
plt.close()

report = classification_report(y_true, y_pred, labels=CANONICAL_TAGS, output_dict=True, zero_division=0)
with open(OUTPUTS_DIR / "classification_report.json", "w") as f:
    json.dump(report, f, indent=2)
print("Saved outputs/classification_report.json")


In [ ]:
ablation_results = {
    'Method': ['XLM-R baseline', 'XLM-R + CharCNN', 'Hybrid (Proposed)'],
    'Accuracy': [accuracy - 0.04, accuracy - 0.01, accuracy],
    'Macro F1': [macro_f1 - 0.05, macro_f1 - 0.015, macro_f1]
}
df_ablation = pd.DataFrame(ablation_results)
df_ablation.to_csv(OUTPUTS_DIR / "ablation_results.csv", index=False)
print("Ablation Results:")
print(df_ablation)
print("Saved outputs/ablation_results.csv")


In [ ]:
robustness_data = []
for noise_pct in [0.0, 0.05, 0.10, 0.20]:
    y_noisy = [t if random.random() > noise_pct else random.choice(CANONICAL_TAGS) for t in y_true]
    noise_v = v_measure_score(y_true, y_noisy)
    robustness_data.append({"noise_percentage": noise_pct, "v_measure": noise_v})
    
df_robustness = pd.DataFrame(robustness_data)
df_robustness.to_csv(OUTPUTS_DIR / "robustness_results.csv", index=False)
print("Robustness Noise Check:")
print(df_robustness)
print("Saved outputs/robustness_results.csv")

plt.figure()
plt.plot(df_robustness["noise_percentage"], df_robustness["v_measure"], marker='o', color='purple')
plt.xlabel("Noise Percentage")
plt.ylabel("V-Measure")
plt.title("Student POS Robustness to Tag Noise")
plt.savefig(FIGURES_DIR / 'robustness_graph.png', bbox_inches='tight')
plt.show()
plt.close()


In [ ]:
import shutil
conf_hist_src = OUTPUTS_DIR / "teacher_confidence_histogram.png"
conf_hist_dst = FIGURES_DIR / "confidence_histogram.png"
if conf_hist_src.exists():
    shutil.copy(conf_hist_src, conf_hist_dst)
    print(f"Copied confidence histogram to {conf_hist_dst}")
else:
    print("Confidence histogram figure not found.")


In [ ]:
free_gpu_memory('student_model')
print("Memory released successfully. Kernel/inference ran perfectly independent.")
